# FASE 3 - Plano 2: EDA Geografica

**Pessoa 2** — Evidencia geografica do Ato 1

Objetivo: Produzir mapa coropletico por UF (bad_review_rate normalizado por taxa, nao volume) e heatmap de rotas origem x destino.

Outputs:
- `data/processed/geo_aggregated.parquet` — agregacao por UF para Streamlit (Phase 6)
- `reports/figures/eda03_choropleth_bad_reviews_uf.png` — mapa estatico para slides
- `reports/figures/eda03_choropleth_static.html` — mapa interativo folium
- `reports/figures/eda05_rotas_heatmap.png` — heatmap corredores origem x destino


## Celula 1 — Setup

In [ ]:

import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
import seaborn as sns
import requests
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURES = PROJECT_ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
DOCS = PROJECT_ROOT / "docs"
DOCS.mkdir(parents=True, exist_ok=True)
GOLD = PROJECT_ROOT / "data" / "gold" / "olist_gold.parquet"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"GOLD exists: {GOLD.exists()}")
print("Setup OK")


## Celula 2 — Load gold e derivacao defensiva dias_atraso

In [ ]:
df = pd.read_parquet(GOLD)

# Derivar dias_atraso defensivamente (formula padrao do contrato)
df["dias_atraso"] = (
    pd.to_datetime(df["order_delivered_customer_date"])
    - pd.to_datetime(df["order_estimated_delivery_date"])
).dt.days

print(f"Shape: {df.shape}")
print(f"customer_state valores unicos: {sorted(df['customer_state'].dropna().unique())}")
print(f"dias_atraso: mean={df['dias_atraso'].mean():.1f}, notna={df['dias_atraso'].notna().sum()}")

## Celula 3 — Agregacao geografica com schema completo

In [ ]:
geo_agg = (
    df.groupby("customer_state")
    .agg(
        total_orders=("order_id", "count"),
        bad_reviews=("bad_review", "sum"),
        avg_dias_atraso=("dias_atraso", "mean"),
        avg_freight_value=("freight_value", "mean"),
    )
    .assign(bad_review_rate=lambda x: x["bad_reviews"] / x["total_orders"])
    .reset_index()
)
print(geo_agg.sort_values("bad_review_rate", ascending=False).head(10))

# Export para Streamlit (Phase 6)
geo_agg.to_parquet(PROCESSED / "geo_aggregated.parquet", index=False)
print(f"Exportado: data/processed/geo_aggregated.parquet ({len(geo_agg)} UFs)")

## Celula 4 — Download e load do GeoJSON do Brasil

In [ ]:

GEOJSON_PATH = DOCS / "brazil_states.geojson"
GEOJSON_URL = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"

if not GEOJSON_PATH.exists():
    r = requests.get(GEOJSON_URL, timeout=30)
    r.raise_for_status()
    GEOJSON_PATH.write_bytes(r.content)
    print(f"GeoJSON baixado: {GEOJSON_PATH}")
else:
    print(f"GeoJSON já existe: {GEOJSON_PATH}")

br_states = gpd.read_file(str(GEOJSON_PATH))
print("Colunas do GeoJSON:", br_states.columns.tolist())


## Celula 5 — Identificar campo de sigla e fazer merge

In [ ]:
# Detectar automaticamente o campo de sigla
possible_fields = ["sigla", "abbrev_state", "UF", "CD_UF", "SIGLA_UF"]
SIGLA_COL = None
for field in possible_fields:
    if field in br_states.columns:
        SIGLA_COL = field
        print(f"Campo de sigla encontrado: {SIGLA_COL}")
        break

if SIGLA_COL is None:
    print("ATENCAO: nenhum campo de sigla padrao encontrado. Colunas:", br_states.columns.tolist())
    raise ValueError("Ajuste SIGLA_COL manualmente para o campo correto")

# Nota: remover colunas com Timestamp antes do merge para compatibilidade com folium
br_states_clean = br_states.drop(columns=[
    c for c in br_states.columns
    if c != 'geometry' and br_states[c].dtype == 'datetime64[ns]'
], errors='ignore')

br_merged = br_states_clean.merge(geo_agg, left_on=SIGLA_COL, right_on="customer_state", how="left")
print(f"Merge: {br_merged['bad_review_rate'].notna().sum()} UFs com dados")

## Celula 6 — Choropleth interativo com folium

In [ ]:

m = folium.Map(location=[-15, -55], zoom_start=4, tiles="CartoDB positron")
folium.Choropleth(
    geo_data=str(GEOJSON_PATH),
    data=geo_agg,
    columns=["customer_state", "bad_review_rate"],
    key_on=f"feature.properties.{SIGLA_COL}",
    fill_color="YlOrRd",
    fill_opacity=0.75,
    line_opacity=0.3,
    legend_name="Taxa de Avaliações 1-2 Estrelas",
    nan_fill_color="lightgray",
).add_to(m)
folium.LayerControl().add_to(m)
m.save(str(FIGURES / "eda03_choropleth_static.html"))
print("Exportado: eda03_choropleth_static.html")


## Celula 7 — Choropleth estatico para slides

> **Nota:** Usa fallback geopandas/matplotlib (kaleido requer Chrome headless, nao disponivel neste ambiente).


In [ ]:

# EDA-03 — Choropleth estático: legível, com labels das UFs e ranking lateral
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(16, 10),
                          gridspec_kw={"width_ratios": [2.5, 1]})
ax_map, ax_rank = axes

# --- Mapa ---
br_merged.plot(
    column="bad_review_rate",
    cmap="YlOrRd",
    legend=False,
    missing_kwds={"color": "#ecf0f1", "label": "Sem dados"},
    ax=ax_map,
    edgecolor="#bdc3c7",
    linewidth=0.5,
)

# Labels das UFs nos centróides
for _, row in br_merged.iterrows():
    sigla = row.get(SIGLA_COL, "")
    rate  = row.get("bad_review_rate", None)
    if sigla and rate is not None and not __import__("numpy").isnan(rate):
        centroid = row.geometry.centroid
        ax_map.text(centroid.x, centroid.y, sigla,
                    ha="center", va="center", fontsize=7.5,
                    fontweight="bold", color="#2c3e50")

# Colorbar manual
cmap = plt.cm.YlOrRd
norm = mcolors.Normalize(vmin=geo_agg["bad_review_rate"].min(),
                          vmax=geo_agg["bad_review_rate"].max())
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax_map, orientation="vertical",
                    fraction=0.03, pad=0.02, shrink=0.7)
cbar.set_label("Taxa de avaliações ruins (1-2 ★)", fontsize=11)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

ax_map.set_title("Onde a logística falha mais?\nTaxa de avaliações ruins por estado (UF)",
                 fontsize=14, fontweight="bold", pad=12)
ax_map.axis("off")

# --- Ranking lateral ---
rank = geo_agg.sort_values("bad_review_rate", ascending=False).head(10).reset_index(drop=True)
rank_colors = [cmap(norm(r)) for r in rank["bad_review_rate"]]

bars = ax_rank.barh(rank["customer_state"][::-1], rank["bad_review_rate"][::-1],
                    color=rank_colors[::-1], edgecolor="white", height=0.65)

for bar, (_, row) in zip(bars, rank[::-1].iterrows()):
    ax_rank.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                 f"{row['bad_review_rate']:.1%}",
                 va="center", fontsize=10, fontweight="bold", color="#2c3e50")

ax_rank.set_title("Top 10 UFs\npor taxa de insatisfação", fontsize=12, fontweight="bold")
ax_rank.set_xlabel("Taxa de avaliações ruins", fontsize=10)
ax_rank.set_xlim(0, rank["bad_review_rate"].max() * 1.25)
ax_rank.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax_rank.spines[["top", "right"]].set_visible(False)

fig.suptitle("Estados do Norte e Nordeste concentram maior insatisfação",
             fontsize=16, fontweight="bold", y=1.01)

fig.tight_layout()
fig.savefig(FIGURES / "eda03_choropleth_bad_reviews_uf.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Exportado: eda03_choropleth_bad_reviews_uf.png")


---

## EDA-05: Heatmap de Rotas Origem x Destino (top-15 origens)

## Celula 8 — Verificar disponibilidade de seller_state

In [ ]:
assert "seller_state" in df.columns, "seller_state ausente na gold table"
assert "dias_atraso" in df.columns, "dias_atraso ausente"

route_data = df.dropna(subset=["dias_atraso", "seller_state", "customer_state"]).copy()
print(f"Linhas com dados de rota: {len(route_data):,} de {len(df):,}")
print(f"seller_state valores unicos: {route_data['seller_state'].nunique()}")

## Celula 9 — Filtrar top-15 origens por volume (evita 27x27 ilegivel)

In [ ]:
top_origins = (
    route_data.groupby("seller_state")["order_id"]
    .count()
    .nlargest(15)
    .index
)
print(f"Top-15 origens (seller_state): {sorted(top_origins.tolist())}")

route_filtered = route_data[route_data["seller_state"].isin(top_origins)]

## Celula 10 — Pivot table de atraso medio por corredor

In [ ]:
pivot = route_filtered.pivot_table(
    values="dias_atraso",
    index="seller_state",
    columns="customer_state",
    aggfunc="mean",
)

print(f"Pivot shape: {pivot.shape} (origens x destinos)")
print(f"Atraso medio geral: {pivot.values[~__import__('numpy').isnan(pivot.values)].mean():.1f} dias")

## Celula 11 — Heatmap seaborn (slide export)

In [ ]:

# EDA-05 — Heatmap de rotas: reduzido a top-10 origens, com piores corredores anotados
# Mensagem: SP→Nordeste/Norte é o corredor mais problemático

import numpy as np

# Filtrar top-10 origens (mais legível que 15)
top_origins_10 = (
    route_data.groupby("seller_state")["order_id"].count()
    .nlargest(10).index
)
route_f10 = route_data[route_data["seller_state"].isin(top_origins_10)]

pivot10 = route_f10.pivot_table(
    values="dias_atraso",
    index="seller_state",
    columns="customer_state",
    aggfunc="mean",
)

fig, ax = plt.subplots(figsize=(18, 7))

sns.heatmap(
    pivot10,
    cmap="RdYlGn_r",
    center=0,
    linewidths=0.4,
    linecolor="white",
    annot=True,
    fmt=".0f",
    annot_kws={"size": 8},
    cbar_kws={"label": "Dias de atraso médio\n(negativo = antecipado)", "shrink": 0.8},
    ax=ax,
)

# Destacar os 3 piores corredores com borda vermelha
worst3 = (
    route_f10.groupby(["seller_state", "customer_state"])["dias_atraso"]
    .agg(["mean", "count"]).reset_index()
    .query("count >= 30")
    .nlargest(3, "mean")
)
for _, row in worst3.iterrows():
    if row["seller_state"] in pivot10.index and row["customer_state"] in pivot10.columns:
        r = pivot10.index.get_loc(row["seller_state"])
        c = pivot10.columns.get_loc(row["customer_state"])
        ax.add_patch(plt.Rectangle((c, r), 1, 1, fill=False,
                                    edgecolor="#c0392b", lw=2.5, zorder=5))

ax.set_title("Dias de atraso médio por corredor logístico (Vendedor → Cliente)",
             fontsize=15, fontweight="bold", pad=14)
ax.text(0.5, 1.01,
        "Vermelho = mais atrasado  |  Verde = mais antecipado  |  □ vermelho = 3 piores corredores",
        transform=ax.transAxes, ha="center", fontsize=10, color="#7f8c8d")
ax.set_xlabel("UF do cliente (destino)", fontsize=12)
ax.set_ylabel("UF do vendedor (origem)", fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=10)

fig.tight_layout()
fig.savefig(FIGURES / "eda05_rotas_heatmap.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Exportado: eda05_rotas_heatmap.png")


## Celula 12 — Top 10 corredores mais problematicos (insight de narrativa)

In [ ]:
corridor_avg = (
    route_filtered.groupby(["seller_state", "customer_state"])["dias_atraso"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "atraso_medio", "count": "volume"})
    .query("volume >= 50")
    .sort_values("atraso_medio", ascending=False)
    .head(10)
)
print("Top 10 corredores mais atrasados (minimo 50 pedidos):")
print(corridor_avg.to_string(index=False))